# Consolidation Exercise 2 — Comparing Algorithms

So far, we've covered:

+ Variables, data types and operators
+ If-then-else
+ Loops (for, while)
+ Functions
+ Lists
+ Input and Output

This week, we'll be doing our second **consolidation** exercise. As a reminder, our aim here is to give you a less structured exercise, closer to what you'll face after this course, and as preparation for the summative assessment at the end of the course. Before diving in and writing code, you should think carefully about how you can break the larger task down into smaller chunks, and how you can use functions to structure and organise your code. As before, we'll provide formative feedback for work submitted via Blackboard. The due date date for this submission is 29th November at 5pm. 


## Introduction

Engineers typically compare the complexity of off-the-shelf algorithms using a combination of:
- theoretical analysis using (Big O)
- measuring actual performance through experiments (benchmarking)
- practical resource requirements (real contraints and data)

The important distinction is that Big O notation tells you how an algorithm scales, but it doesn't necessarily tell you which algorithm will perform best in a real engineering application.

## Task

You are developing an autonomous mobile robot for use in a warehouse. 

The robot needs to recognise obstacles using an array of distance sensors. 

A simple array could include 3 distance sensors and record three distance measurements per observation. 

In contrast, a depth camera is equivalent to multiple distance sensors, and could record hundereds to thousands of distance measurements per observation.

<img src="https://github.com/engmaths/SEMT10002_2026/blob/main/media/week_10/robots_with_sensors.png?raw=true" width="60%">

To classify a new observation, the observation should be compared with some existing data that has already been classified. This data, labelled with the correct classification is called the training data. 

Here is an example for a robot with 3 sensors:

| Left (cm) | Centre (cm) | Right (cm) | Classification |
|-----------|-------------|------------|----------------|
| 100 | 100 | 100 | Clear |
| 80 | 90 | 85 | Clear |
| 20 | 100 | 100 | Obstacle left |
| 15 | 30 | 100 | Obstacle left |
| 100 | 20 | 100 | Obstacle ahead |
| 100 | 15 | 25 | Obstacle ahead |
| 100 | 100 | 20 | Obstacle right |
| 90 | 100 | 15 | Obstacle right |
| 10 | 15 | 20 | Surrounded |

The input size is $n$ observations $\times$ $m$ sensors 

This exercise has three parts in which you will compare algorithms for recognising and classifying obstacles using:
1. **Algorithmic complexity**: How does the computational effort grow as the input size increases?
2. **Computational performance (benchmarking)**: How long does the algorithm actually take to execute, and how much memory does it use?
3. **Engineering suitability**: Is the algorithm appropriate for the constraints of the application?

# K-nearest neighbours (k-NN) 

k-NN is a simple machine learning algorithm used to classify new observations based on previously collected data.

For example, an autonomous robot might use sensor measurements to recognise objects. It has a dataset of previously observed objects, each labelled with a category such as box, person or pallet. The algorithm is called k-nearest neighbours because it uses the k closest observations to make its prediction.

When the robot encounters a new object, k-NN:
1. Calculates the distance between the new observation and every observation in its training dataset.
2. Identifies the k most similar (nearest) observations.
3. Predicts the category by selecting the most common label among those neighbours.

<img src="https://github.com/engmaths/SEMT10002_2026/blob/main/media/week_10/KNN.png?raw=true" width="50%">

Below are two different implementations of kNN that produce the same predictions but use different approaches to identify the nearest neighbours.

### KNN A
This implementation:
- A. Calculates the distance from the new observation to every training observation
- B. Maintains a sorted list containing only the k closest observations.
- C. Counts the labels of the k nearest neighbours

In [ ]:
def knn_A(training_data, new_observation, k):

    """
    Maintains a sorted list containing only the k closest observations
    """

    nearest = []

    for features, label in training_data:

        # Step A: Calculate distance to every training observation
        distance = 0
        for i in range(len(new_observation)):
            distance += (features[i] - new_observation[i])**2

        # Step B: Maintains a sorted list containing only the k closest observations.
        # Add the new observation to the end of the list of closest observations
        nearest.append((distance, label))

        # Gets the index of the new observation in the list
        index = len(nearest) - 1

        # Work backwards through the list checking if each item is smaller than the item 
        # before it and swapping if it is, until all items have been checked
        while index > 0 and nearest[index-1][0] > nearest[index][0]:
            nearest[index], nearest[index-1] = nearest[index-1], nearest[index]
            index -= 1

        # Keep only the k nearest neighbours
        if len(nearest) > k:
            nearest.pop()

    # Step C: Count the labels of the k nearest neighbours
    votes = {}
    for distance, label in nearest:
        votes[label] = votes.get(label, 0) + 1

    # Find the most common label amongst the nearest neighbours
    # Initialise most common label as first in dictionary
    most_common_label = list(votes.keys())[0]

    # Return the most common label
    for label in votes:
        if votes[label] > votes[most_common_label]:
            most_common_label = label

    return most_common_label
    

### k-NN B

This implementation:
- A. Calculates the distance from the new observation to every training observation
- B. Sorts all the distances using the bubble sort algorithm
- C. Counts the labels of the k nearest neighbours

Bubble sort repeatedly compares adjacent elements and swaps them if they are in the wrong order.

<img src="https://github.com/engmaths/SEMT10002_2026/blob/main/media/week_10/bubble_sort.png?raw=true" width="35%">


In [ ]:
def knn_B(training_data, new_observation, k):
    """
    Calculates the distance from the new observation to every training observation
    Sorts all the distances using bubble sort
    Counts the labels of the k nearest neighbours
    Returns the most common label
    """
    distances = []

    # Calculate distance to every training observation
    for features, label in training_data:
        distance = 0
        for i in range(len(new_observation)):
            distance += (features[i] - new_observation[i])**2
        distances.append((distance, label))

    # Sort all distances using bubble sort
    n = len(distances)
    for i in range(n-1):
        for j in range(0, n-i-1):
            if distances[j][0] > distances[j+1][0]:
                distances[j], distances[j+1] = distances[j+1], distances[j]

    # Count the labels of the k nearest neighbours
    votes = {}
    for i in range(k):
        label = distances[i][1]
        votes[label] = votes.get(label, 0) + 1

    # Find the most common label amongst the nearest neighbours
    # Initialise most common label as first in dictionary
    most_common_label = list(votes.keys())[0]

    # Return the most common label
    for label in votes:
        if votes[label] > votes[most_common_label]:
            most_common_label = label

    return most_common_label


The following example tests both functions with a smal sample data set - each produces the correct result
| Left (cm) | Centre (cm) | Right (cm) | Classification |
|-----------|-------------|------------|----------------|
| 100 | 100 | 100 | Clear |
| 80 | 90 | 85 | Clear |
| 20 | 100 | 100 | Obstacle left |
| 15 | 30 | 100 | Obstacle left |
| 100 | 20 | 100 | Obstacle ahead |
| 100 | 15 | 25 | Obstacle ahead |
| 100 | 100 | 20 | Obstacle right |
| 90 | 100 | 15 | Obstacle right |
| 10 | 15 | 20 | Surrounded |

In [36]:
sample_data = [
    ([100, 100, 100], "clear"),
    ([80, 90, 85],    "clear"),
    ([20, 100, 100],  "obstacle left"),
    ([15, 30, 100],   "obstacle left"),
    ([12, 34, 90],    "obstacle left"),
    ([90, 92, 95],    "clear"),
    ([112, 89, 102],  "clear"),
    ([17, 70, 95],    "obstacle left")
]

new_observation = [21, 24, 90]

print(knn_A(sample_data, new_observation, 3))

print(knn_B(sample_data, new_observation, 3))

obstacle left
obstacle left


## Getting started

We've provided you with a file `comparing_algorithms.py` to write your code in and `README.txt` to write your analysis in. 

- Part 1 (Algorithmic Compexity) - write your answers in `README.txt`
- Part 2 (Computational performance) - in `comparing_algorithms.py` you should update and run the `main` function.
- Part 3 (Computational performance) - edit `comparing_algorithms.py` and write in `README.txt`

In `comparing_algorithms.py` you should update and run the `main` function. Think carefully about how you can break the larger task down into smaller chunks, and how you can use functions to structure and organise your code.

## Part 1 - Algorithmic Complexity

Write your answers in `README.txt`

Analyse the complexity of algorithms k-NN A and k-NN B: 
1. Analyse complexity of each step within the algorithm individually (calculate distances, sort results, store data)
2. Add complexity of sequential operations 
3. Simplify to the dominant term to get the complexity of the algorithm

***

## Example: Complexity of k-NN A

### 1. Analyse complexity of each step within the algorithm

##### **Complexity of Step A:**
```Python
#  Step A: Calculate distance to every training observation
for features, label in training_data:

    distance = 0

    for i in range(len(new_observation)):
        distance += (features[i] - new_observation[i])**2
```
**Time complexity is $O(nm)$**: Grows linearly with number of observations $n$ (outer loop), and grows linearly with number of senosrs $m$ (inner loop)<br>
 **Space complexity is $O(1)$**: Size of `distance` is constant for any number of observations $n$

##### **Complexity of step B:**
```Python
nearest = []

for features, label in training_data:

    ...

    # Step B: Maintains a sorted list containing only the k closest observations.
    # Add the new observation to the end of the list of closest observations
    nearest.append((distance, label))

    # Gets the index of the new observation in the list
    index = len(nearest) - 1

    # Use insertion sort (for final list element only) to move it backwards to the correct position
    while index > 0 and nearest[index-1][0] > nearest[index][0]:
        nearest[index], nearest[index-1] = nearest[index-1], nearest[index]
        index -= 1

    # Keep only the k nearest neighbours
    if len(nearest) > k:
        nearest.pop()
```
**Time complexity is $O(n)$**: Grows linearly with number of observations $n$ (outer loop). <br>Inner while loop has constant complexity for any value of $n$ and $m$ (*but* grows linearly with k (number of nearest neighbours used))

**Space complexity is $O(1)$**: Constant complexity for any value of $n$ and $m$ <br>(*but* `nearest` grows linearly with k (number of nearest neighbours used))

##### **Complexity of step C:**
```Python
# Step C: Count the labels of the k nearest neighbours
votes = {}
for distance, label in nearest:
    votes[label] = votes.get(label, 0) + 1

# Find the most common label amongst the nearest neighbours
# Initialise most common label as first in dictionary
most_common_label = list(votes.keys())[0]

# Return the most common label
for label in votes:
    if votes[label] > votes[most_common_label]:
        most_common_label = label

return most_common_label
```
**Time complexity is $O(1)$**: Constant for any value of $n$ and $m$ <br>(*but* grows linearly with k (number of nearest neighbours used))

**Space complexity is $O(1)$**: Constant for any value of $n$ and $m$ <br>(*but* grows linearly with k (number of nearest neighbours used))

***

### 2. Add complexity of sequential operations to get complexity of the algorithm

**Time complexity** = $\underbrace{O(nm)}_{\text{Step A}} + \underbrace{O(n)}_{\text{Step B}} + \underbrace{O(1)}_{\text{Step C}}$

**Space complexity** = $\underbrace{O(1)}_{\text{Step A}} + \underbrace{O(1)}_{\text{Step B}} + \underbrace{O(1)}_{\text{Step C}}$
***

### 3. Simplify to the dominant term

**Time complexity of the algorithm** = $O(nm)$ <br>
($O(nm)$ is the highest order $n$ term and the highest order $m$ term)

**Space complexity of the algorithm** = $O(1)$<br>
($O(1)$ is the highest order term

***

## Complexity of k-NN B

Solve this for each step of the algorithm and note your answers down in `README.txt`

### 1. Analyse complexity of each step within the algorithm

##### **Complexity of  step A:**
```Python
distances = []

#  Step A: Calculate distance to every training observation
for features, label in training_data:

    distance = 0

    for i in range(len(new_observation)):
        distance += (features[i] - new_observation[i])**2
    distances.append((distance, label))
```
**Time complexity is $O(nm)$**: Grows linearly with number of observations $n$ (outer loop), and grows linearly with number of sensors $m$ (inner loop)<br>
 **Space complexity is $O(n)$**: `distances` grows linearly with number of observations $n$





##### **Complexity of  step B:**
```Python
# Step B: Sort all distances using bubble sort
n = len(distances)
for i in range(n-1):
    for j in range(0, n-i-1):
        if distances[j][0] > distances[j+1][0]:
            distances[j], distances[j+1] = distances[j+1], distances[j]
```
**Time complexity is $O(n^2)$**: Grows quadratically with number of observations $n$ (number of comparisons is $k = \frac{n(n-1)}{2}$ - same as finding repeats example from Algorithmic Complexity seminar)<br>
**Space complexity is $O(1)$**: No additional variables created

##### **Complexity of step C:**
```Python
# Step C: Count the labels of the k nearest neighbours
votes = {}
for i in range(k):
    label = distances[i][1]
    votes[label] = votes.get(label, 0) + 1

# Find the most common label amongst the nearest neighbours
# Initialise most common label as first in dictionary
most_common_label = list(votes.keys())[0]

# Return the most common label
for label in votes:
    if votes[label] > votes[most_common_label]:
        most_common_label = label

return most_common_label
```
**Time complexity is $O(1)$**: Constant for any value of $n$ and $m$ <br>(*but* grows linearly with k (number of nearest neighbours used))<br>

**Space complexity is $O(1)$**: Constant for any value of $n$ and $m$ <br>(*but* grows linearly with k (number of nearest neighbours used))
***




### 2. Add complexity of sequential operations to get complexity of the algorithm*

**Time complexity** = $\underbrace{O(nm)}_{\text{Step A}} + \underbrace{O(n^2)}_{\text{Step B}} + \underbrace{O(1)}_{\text{Step C}}$

**Space complexity** = $\underbrace{O(n)}_{\text{Step A}} + \underbrace{O(1)}_{\text{Step B}} + \underbrace{O(1)}_{\text{Step C}}$


***

### 3. Simplify to the dominant term

**Time complexity of the algorithm** = $O(n^2) + O(nm)$ <br>
($O(n^2)$ is the highest order $n$ term and $O(nm)$ is the highest order $m$ term)

**Space complexity of the algorithm*** = $O(n)$<br>
($O(n)$ is the highest order $n$ term 

***

## Part 2 - Computational performance (benchmarking)

In `comparing_algorithms.py` you should update and run the `main` function. Think carefully about how you can break the larger task down into smaller chunks, and how you can use functions to structure and organise your code.

The file `obstacle_avoidance_data.csv` contains 20,000 labelled observations from three distance sensors (left, centre, right), similar to the exampe table shown below

| Left (cm) | Centre (cm) | Right (cm) | Classification |
|-----------|-------------|------------|----------------|
| 100 | 100 | 100 | Clear |
| 80 | 90 | 85 | Clear |
| 20 | 100 | 100 | Obstacle left |
| 15 | 30 | 100 | Obstacle left |
| 100 | 20 | 100 | Obstacle ahead |
| 100 | 15 | 25 | Obstacle ahead |
| 100 | 100 | 20 | Obstacle right |
| 90 | 100 | 15 | Obstacle right |
| 10 | 15 | 20 | Surrounded |

The data set can grows linearly with the number of observations $n$ and/or the number of sensors $m$.<br>
A larger number of sensor $m$ gives higher resolution information about the environment which can improve the accuracy of obstacke detection.<br>
More training observations $n$ are generally desirable because they give the algorithm more examples of different situations it might encounter.<br>
However, there is a trade-off between classification performance and computational resources.

Your task is to compare the computational performance (execution time and memory usage) of the two algorithms as the size of the input data increases, by using the data to classify a randomly generated new observation.

For the benchmarking exercise, we will consider the growth of the time and space as a function of $n$ only (ignore $m$ terms)

Investigate a minimum of **3 values of $n$** observations.

Start by using **k=5** nearest neightbours

1. Import the data from `obstacle_avoidance_data.csv`
1. Measure execution time of each algorithm in seconds with `time.perf_counter()` (example below)
1. Measure memory use of each algorithm with `tracemalloc.start()` (example below) [Note: Measureing memory takes a long time, so write your program and test with small values of $n$ before testing largeer values of $n$]
    - Perform time and space measurements separately, and exclude print statements, to avoid distorting the measurements. 
    - Use command line arguments and flags to configure your program to measure either k-NN A or k-NN B  when the program runs 
    - Repeat measurements at least three times and calculate the average (execution time can vary due to other processes running on the computer, background activity and other sources of measurement noise).
    - Perform experiments on the same computer as the computer you use effects the execultion time 
1. Record your results as a table of the format shown below in a csv file for each algorithm where:
    - Approximate number of operations is predicted using Big O time complexity e.g. number of operations = $O(n)=1000$, if $n=1000$ 
    - Approximate data stored is predicted using Big O space complexity e.g. data stored = $O(n)=1000$, if $n=1000$ 
    - Time is recorded using `time.perf_counter()`
    - Memory is recorded using `tracemalloc.start()`
1. In the README.txt file, briefly analyse your results:
    - Describe how increasing $n$ affects each implementation
    - How does the measured computational performance compare with the predicted (Big O) relationships between in put size $n$ and time/space required by the algorithm? 

#### Results format
| n | approximate number of operations | time (seconds) | seconds/comparison | approximate size of data stored | memory (KB) | KB/data |
|-----------|-------------|------------|----------------|----------------|----------------|----------------|
| 10,000     |  |  |  ||||
| ... |  |  |  ||||
| ... |  | |  ||||


The following example shows how to time the execution of a section of code:

In [37]:
import time 

start_time = time.perf_counter()

knn_A(sample_data, new_observation, 3)

end_time = time.perf_counter()

execution_time = end_time - start_time
print(f'Algorithm A took {execution_time} seconds' )

Algorithm A took 3.450002986937761e-05 seconds


The following example shows how to measure the space taken by a secion of code:

In [38]:
import tracemalloc

tracemalloc.start()

knn_A(sample_data, new_observation, 3)

current, peak = tracemalloc.get_traced_memory()

tracemalloc.stop()

print(f'Peak memory usage = {peak / 1024} KB')

Peak memory usage = 20.6728515625 KB


## Part 3 - Engineering suitability

Edit `comparing_algorithms.py` and write in `README.txt`

Here are some real-world constraints that the robot obstacle avoidance needs to satisfy

| Parameter | Engineering constraint | Why it matters |
|---|---|---|
| Available RAM | 264 KB | A small microcontroller has very limited working memory. |
| Maximum RAM allocated to KNN | 32 KB | Leaves the majority of RAM available for sensor acquisition, motor control, communication, etc |
| Maximum classification time | 10 ms | Leaves time for motor control and other tasks. |
| Training dataset | Up to 10,000 observations | Beyind 10,000 observations observations the improvements in accuracy have been <br>shown to be negligible for this particuar application  |

Have a look at your experimental results. 

Which algorithm(s) with which values of $n$ meet the engineering contraints?

Given that the time complexity is a function of the number of observations $n$ and the number of sensors $m$, which combinations woudl meet the requirements if the number of sensors was increased to a)10 b)100 c) 1000?